In [ ]:
from src import analysis, common

TASK = "steady_flow"
DATASET_ROOT = common.paths.get_dataset_root()
RUN_NAMES = [
    "replace_with_current_run_name_a",
    "replace_with_current_run_name_b",
]
RUN_LABELS = {
    "replace_with_current_run_name_a": "Run A",
    "replace_with_current_run_name_b": "Run B",
}
SHOW_ID_EVALUATION = True
SHOW_OOD_EVALUATION = True

In [ ]:
# Build artifacts through the public service; dataframe validation owns comparability.
datasets_eval_id = {}
datasets_eval_ood = {}
for run_name in RUN_NAMES:
    run_dir = common.paths.resolve_run_output_dir(TASK, run_name)
    frames_by_run = analysis.artifact_service.build_artifacts(
        runs_root=run_dir,
        dataset_root=DATASET_ROOT,
        max_cases=None,
        batch_size=1,
        device_policy="cpu",
        rebuild=False,
    )
    roles = frames_by_run[run_dir.name]
    label = RUN_LABELS.get(run_name, run_name)
    datasets_eval_id[label] = analysis.evaluation.dataframe.build_eval_df(roles["eval"])
    datasets_eval_ood[label] = analysis.evaluation.dataframe.build_eval_df(roles["ood"])

if SHOW_ID_EVALUATION:
    analysis.evaluation.dataframe.validate_comparison(datasets_eval_id, require_physics=True)
if SHOW_OOD_EVALUATION:
    analysis.evaluation.dataframe.validate_comparison(datasets_eval_ood, require_physics=True)

In [ ]:
if SHOW_ID_EVALUATION:
    panel_id = analysis.evaluation.panel.build_evaluation_panel(
        datasets_eval=datasets_eval_id,
        title="Run Comparison ID",
        sections="all",
    )
    display(panel_id)

if SHOW_OOD_EVALUATION:
    panel_ood = analysis.evaluation.panel.build_evaluation_panel(
        datasets_eval=datasets_eval_ood,
        title="Run Comparison OOD",
        sections="all",
    )
    display(panel_ood)